# SARF AraBERT — E1

**Owner:** A  
**Condition:** E1  
**CPT assignment:** general_3k_final_v2.csv (3,000 rows).

CPT on the approved 3K general Saudi-style corpus, then identical MSA fine-tuning.

This notebook contains no model-training implementation. It invokes the one shared `arabert_pipeline.py`, which reads the one shared AraBERT config and the one global 77-label mapping from the project Drive. The held-out Saudi test is not loaded in this notebook.


## 1. Install the fixed environment

Choose a GPU runtime before running this cell. If Colab asks to restart after installation, restart the runtime, then rerun this cell and the following cells in order.


In [1]:
%pip install --upgrade --prefer-binary "tokenizers==0.21.4" "transformers==4.48.3" "accelerate==1.2.1" "datasets==3.2.0" "scikit-learn==1.6.1" "sentencepiece>=0.2.1"
%pip install --upgrade "PyArabic==0.6.15" "farasapy==0.1.1" "emoji==1.4.2"
%pip install --upgrade --no-deps "arabert==1.0.1"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 115.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.4/336.4 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 15.2 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.12.0
    Uninstalling fsspec-2025.12.0:
      Successfully uninstalled fsspec-2025.12.0
  Attempting uninstall: dill
    Found existing installation: dill 0.4.1
    Uninstalling dill-0.4.1:
      Success

## 2. Mount Drive and verify the shared AraBERT source

The project Drive must contain exactly one shared pipeline, config, and label mapping. This cell does not load training data, CPT data, or the held-out test.


In [2]:
from google.colab import drive
from pathlib import Path
import json
import torch

drive.mount("/content/drive", force_remount=False)

PROJECT_ROOT = Path("/content/drive/MyDrive/SARF_BANKING_NLP_PROJECT")
PIPELINE_PATH = PROJECT_ROOT / "src_04" / "arabert_pipeline.py"
CONFIG_PATH = PROJECT_ROOT / "02_configs" / "arabert_config.json"
MAPPING_PATH = PROJECT_ROOT / "02_configs" / "arabert_label_mapping.json"

assert torch.cuda.is_available(), "GPU required; do not run AraBERT training on CPU."
assert PROJECT_ROOT.is_dir(), f"Project folder not found: {PROJECT_ROOT}"
assert PIPELINE_PATH.is_file(), f"Pipeline not found: {PIPELINE_PATH}"
assert CONFIG_PATH.is_file(), f"Config not found: {CONFIG_PATH}"
assert MAPPING_PATH.is_file(), f"Mapping not found: {MAPPING_PATH}"

config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
mapping = json.loads(MAPPING_PATH.read_text(encoding="utf-8"))

assert config["owner"] == "A"
assert config["official_conditions"] == ["E0", "E1", "E2", "E3", "EB"]
assert config["official_seeds"] == [42, 123, 2026]
assert mapping["owner"] == "A"
assert mapping["num_labels"] == 77
assert len(mapping["label_to_id"]) == 77

print({
    "status": "shared AraBERT source verified",
    "owner": config["owner"],
    "condition": "E1",
    "gpu": torch.cuda.get_device_name(0),
    "pipeline": str(PIPELINE_PATH),
    "config": str(CONFIG_PATH),
    "mapping": str(MAPPING_PATH),
    "official_seeds": config["official_seeds"],
    "num_labels": mapping["num_labels"],
    "cpt_assignment": config["conditions"]["E1"],
})


Mounted at /content/drive
{'status': 'shared AraBERT source verified', 'owner': 'A', 'condition': 'E1', 'gpu': 'Tesla T4', 'pipeline': '/content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/src_04/arabert_pipeline.py', 'config': '/content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/02_configs/arabert_config.json', 'mapping': '/content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/02_configs/arabert_label_mapping.json', 'official_seeds': [42, 123, 2026], 'num_labels': 77, 'cpt_assignment': {'description': 'CPT on 3K general Saudi-style text, then identical MSA fine-tuning.', 'cpt_enabled': True, 'cpt': {'corpus_directory_relative_path': '03_synthetic_data/05_final_corpora', 'manifest_relative_path': '03_synthetic_data/06_final_audit/corpus_manifest_v2.json', 'corpus_files': ['general_3k_final_v2.csv'], 'corpus_file_rows': [3000], 'total_rows': 3000}}}


## 3. CPT technical check — required once before official E1

Run this once before `E1_seed_42`. It uses only the first 128 rows of `general_3k_final_v2.csv`, its `text` column only, and five masked-language-model steps. It is **not** E1 and is not part of the 15 official runs.

The temporary checkpoint is saved and reloaded locally to prove the CPT path, then deleted. Its small audit record is saved to `05_runs/arabert/dev/cpt_technical_check/`. Do not rerun after it has succeeded.


In [3]:
!python "{PIPELINE_PATH}" --project-root "{PROJECT_ROOT}" --mode cpt_technical_check


2026-09-12 06:03:04.771137: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
tokenizer_config.json: 100% 611/611 [00:00<00:00, 4.18MB/s]
config.json: 100% 384/384 [00:00<00:00, 3.20MB/s]
vocab.txt: 720kB [00:00, 31.3MB/s]
tokenizer.json: 2.31MB [00:00, 149MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 733kB/s]
Tokenizing fixed E1 CPT technical subset: 100% 128/128 [00:00<00:00, 6134.83 examples/s]
model.safetensors: 100% 543M/543M [00:06<00:00, 85.5MB/s]
BertForMaskedLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability t

## 4. Run E1 one seed at a time

Run only one seed per execution. The approved sequence within this condition is `42`, then `123`, then `2026`. The shared pipeline creates outputs directly under `05_runs/arabert/E1/seed_<seed>` and `06_models_and_checkpoints/arabert/E1/seed_<seed>`, refusing to overwrite a completed run.


In [4]:
SEED = 42
assert SEED in [42, 123, 2026]

!python "{PIPELINE_PATH}" --project-root "{PROJECT_ROOT}" --condition "E1" --seed {SEED}


2026-09-12 06:04:19.592365: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'farasa-api.qcri.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
100% 241M/241M [07:22<00:00, 546kiB/s]
[2026-09-12 06:11:50,194 - farasapy_logger - WARNING]: Be careful with large lines as they may break on interactive mode. You may switch to Standalone mode for such cases.
Map: 100% 10732/10732 [00:00<00:00, 13853.70 examples/s]
Map: 100% 1229/1229 [00:00<00:00, 13119.71 examples/s]
Tokenizing approved E1 CPT corpus: 100% 30

In [5]:
SEED = 123
assert SEED in [42, 123, 2026]

!python "{PIPELINE_PATH}" --project-root "{PROJECT_ROOT}" --condition "E1" --seed {SEED}


2026-09-12 06:30:55.586235: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
[2026-09-12 06:31:02,247 - farasapy_logger - WARNING]: Be careful with large lines as they may break on interactive mode. You may switch to Standalone mode for such cases.
Map: 100% 10732/10732 [00:00<00:00, 13892.02 examples/s]
Map: 100% 1229/1229 [00:00<00:00, 12267.87 examples/s]
Tokenizing approved E1 CPT corpus: 100% 3000/3000 [00:00<00:00, 13790.70 examples/s]
BertForMaskedLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and ot

In [6]:
SEED = 2026
assert SEED in [42, 123, 2026]

!python "{PIPELINE_PATH}" --project-root "{PROJECT_ROOT}" --condition "E1" --seed {SEED}


2026-09-12 06:46:22.565888: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
[2026-09-12 06:46:29,748 - farasapy_logger - WARNING]: Be careful with large lines as they may break on interactive mode. You may switch to Standalone mode for such cases.
Map: 100% 10732/10732 [00:01<00:00, 8357.35 examples/s]
Map: 100% 1229/1229 [00:00<00:00, 7575.21 examples/s]
Tokenizing approved E1 CPT corpus: 100% 3000/3000 [00:00<00:00, 8085.45 examples/s]
BertForMaskedLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other